# Quantum circuits
$
\newcommand{\ket}[1]{\left|#1\right\rangle}
\newcommand{\bra}[1]{\left\langle #1\right|}
\newcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle}
\newcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}
$

In [1]:
# allows us to have visibility on our package without installing it in editing mode
import sys;
if ".." not in sys.path: sys.path.append("..")

import itertools
import numpy as np
from qiskit.circuit import QuantumCircuit, Gate
from monaqa2.qiskit.utils_numpy import kron, ketbra, bra, ket
from monaqa2.qiskit.utils_qiskit import get_unitary

Consider an MCMC on $n$ Ising spins, so the configuration space has size $2^n$; its transition matrix $P_{yx}\in\mathbb{R}^{2^n\times 2^n}$ is induced by a trial-move matrix $T_{yx}\in\mathbb{R}^{2^n\times 2^n}$ and by the Metropolis-Hastings acceptance probability $A_{yx}$, with accepted off-diagonal moves weighted by $T_{yx}A_{yx}$ and the rejected probability placed on the diagonal.

The quantum walk circuit is an operator acting on the Hilbert space $\mathcal{H}_a \otimes \mathcal{H}_b \otimes \mathcal{H}_c$, where $\mathcal{H}_a$ and $\mathcal{H}_b$ are two copies of the system register holding the MCMC state, and $\mathcal{H}_c$ is an auxiliary register called the coin. The walk operator and the transition matrix $P$ have related spectrum, so that a filter on the eigenphases on the walk operator corresponds to a filter on the eigenphases of the transition matrix; in this way, one can extract the stationary component of the walk. 

The walk operator has the form

$$
W = R_0 V^\dagger B^\dagger F B V.
$$

where
* $
V\ket{x}_a\ket{0}_b
=
\ket{x}_a
\sum_y \sqrt{T_{yx}}\ket{y}_b
$ is the trial-move unitary. The quantity $T_{yx}$ is nonzero when the trial move from state $x$ to state $y$ is allowed with nonzero proposal probability.
* $
B\ket{x}_a\ket{y}_b\ket{0}_c
=
\ket{x}_a\ket{y}_b
\left(
\sqrt{A_{yx}}\ket{0\cdots 0}_c
+
\sqrt{1-A_{yx}}\ket{\bot_{xy}}_c
\right),
$ is the Boltzmann coin, here $A_{yx}$ is the probability of accepting the trial move from $x$ to $y$, determined by the energy difference between the two states and the inverse temperature; and the state $\ket{\bot_{xy}}_c$ is orthogonal to $\ket{0\cdots 0}_c$.
* $
\begin{align*}
F\ket{x}_a\ket{y}_b\ket{0\cdots 0}_c
&=
\ket{y}_a\ket{x}_b\ket{0\cdots 0}_c, \\
F\ket{x}_a\ket{y}_b\ket{\bot_{xy}}_c
&=
\ket{x}_a\ket{y}_b\ket{\bot_{xy}}_c.
\end{align*}
$ is denoted as the accept-path swap unitary.
* $R_0$ acts as the identity on $\mathcal{H}_a$ and as a reflection about $\ket{0\cdots 0}_b\ket{0\cdots 0}_c$ on $\mathcal{H}_b \otimes \mathcal{H}_c$.

We expect three properties to hold.

1. $U = V^\dagger B^\dagger F B V$ is a block encoding of $X$, the discriminant matrix of the MCMC:

    $$
    (\mathbb{I}_a \otimes \bra{0}_b \otimes \bra{0}_c)
    U
    (\mathbb{I}_a \otimes \ket{0}_b \otimes \ket{0}_c)
    =
    X.
    $$

    For a reversible Markov chain with transition matrix $P$ and stationary distribution $\pi$, the discriminant has entries $X_{yx}=\sqrt{P_{xy}P_{yx}}$; $X$ is similar to $P$ and therefore they share the same spectrum. 
2. The stationary distribution is encoded in the zero-ancilla invariant subspace. In particular, detailed balance implies that the coherent stationary state $\ket{\sqrt{\pi}}=\sum_x \sqrt{\pi_x}\ket{x}$ is a $+1$ eigenvector of $X$: $X\ket{\sqrt{\pi}} = \ket{\sqrt{\pi}}$. Therefore, because $U$ block-encodes $X$, the full state $\ket{\sqrt{\pi}}_a\ket{0}_b\ket{0}_c$ is a $+1$ eigenstate of $U$ with no leakage outside the zero subspace.
3. If $\lambda \in \mathrm{spec}(X)$, then the corresponding walk eigenvalues are $\exp(\pm i \arccos(\lambda)) \in \mathrm{spec}(W)$.

This notebook shows how the quantum walk operator is implemented. 

## Lattice surgery model

## Summary of the resources needed

| Component | Non-clifford depth | Total qubits | Auxiliary qubits |
|---|---|---|---|
| Proposal with uniform move | $0$ | $2n$ | $0$ | 
| Proposal with local spin flip move | $\log_2()$ | $2n$ | $0$ |
| Proposal with quantum-enhanced move (Trotterized with $m$ layers) | $1 + m(2+n)$ | $2n$ | $0$ |
| Reflection |
| Accept path |

## Boltzmann coin

The Boltzmann coin is the most expensive portion of the walk circuit. This is in charge of calculating whether to accept the move and must implement the necessary arithmetic to determine that. There are a few different ways one can implement the Boltzmann coin, here we show three of them. 

* **Fully-phase arithmetic**: this is the most space-efficient way. We define an Hamiltonian acting on $2n$ qubits containing on the diagonal the difference of energies between the two configurations. A polynomial transformation of the desired function get us closer to the (square root of) the acceptance rule. This takes $O(n)$ space and $O(n^2)$ depth of the circuit
* **Hybrid fixed point/phase arithmetic**: a portion of fixed point arithmetic is used to determine the energy difference, this takes $O(n^2)$ space and $O(\log_2 n)$ depth. Then, a polynomial transformation of a many body Hamiltonian allows us to implement the square exponential in the acceptance rule, this takes $O(n)$ space and $O(\log_2 n)$ depth.
* **Fully-fixed point arithmetic**: the non-linear transformation is calculated in fixed point, too. This has comparable space and depth asymptotics of the hybrid arithmetic albeit with worse prefactors, therefore is not tested. 

We remark that the arithmetic can calculate the acceptance rule for any moves without neither locality nor sparsity of the bitflip restrictions. In presence of Ising models with additional constraints (sparsity, bounded degree, ...) one could implement much faster arithmetic likes the ones shown in [..]. However, the generality of the SK model does not allow for such optimizations. Furthermore, a local move (e.g. single spin flip) may exploit the locality of the move to make the delta energy dependent on the $O(n)$ terms affected by the flip instead of the $O(n^2)$. The depth of the addition arithmetic, which is logarithmic in $n$, makes such optimization negligible (also, the local spin flip move is not well behaved in the low temperature regime and therefore less interesting than denser moves). 
